# Loan Default Prediction: Results Presentation

**Predictive modelling, economic decision analysis, and SHAP interpretability for home-equity loan default risk**

This notebook presents the final results from the loan default prediction project. It does not retrain the models. Instead, it loads the saved outputs generated by the project scripts and explains the main analytical findings in a clear, portfolio-ready format.


## Project Overview

The project uses the HMEQ home-equity loan dataset to predict whether a borrower is likely to default or become severely delinquent. The target variable is `BAD`:

- `BAD = 1`: borrower defaulted or was severely delinquent
- `BAD = 0`: borrower repaid the loan

The goal is not only to compare predictive models, but also to translate predictions into lending decisions. This means evaluating the cost of approving risky borrowers and the opportunity cost of rejecting good borrowers.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display, Markdown

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "outputs").exists() and (PROJECT_ROOT.parent / "outputs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.options.display.float_format = "{:,.4f}".format


def load_table(relative_path, title=None, sort_by=None, ascending=False, n=None, columns=None, drop_source=True):
    """Load a result table if it exists; clean saved index/path columns before display."""
    path = PROJECT_ROOT / relative_path
    if not path.exists():
        display(Markdown(f"**Unavailable:** `{relative_path}` was not found."))
        return pd.DataFrame()

    table = pd.read_csv(path)
    table = table.loc[:, ~table.columns.str.match(r"^Unnamed")]
    if drop_source and "source" in table.columns:
        table = table.drop(columns=["source"])
    if columns is not None:
        table = table[[col for col in columns if col in table.columns]]
    if sort_by is not None and sort_by in table.columns:
        table = table.sort_values(sort_by, ascending=ascending)
    if n is not None:
        table = table.head(n)
    if title:
        display(Markdown(f"**{title}**"))
    display(table)
    return table


def display_image(relative_path, title=None, width=None):
    """Display a saved plot if it exists; otherwise continue gracefully."""
    path = PROJECT_ROOT / relative_path
    if not path.exists():
        display(Markdown(f"**Unavailable:** `{relative_path}` was not found."))
        return
    if title:
        display(Markdown(f"**{title}**"))
    display(Image(filename=str(path), width=width))


def get_test_rows(table, preferred_model_name=None):
    """Return test rows from a saved model-comparison table."""
    if table.empty:
        return table
    result = table.copy()
    if "split" in result.columns:
        result = result[result["split"].astype(str).str.lower().eq("test")]
    if preferred_model_name is not None and "model" in result.columns:
        preferred = result[result["model"].eq(preferred_model_name)]
        if not preferred.empty:
            result = preferred
    return result


## Business Problem

A lender faces two types of decision error:

- **False negative:** the model approves a borrower who later defaults. This exposes the lender to credit losses.
- **False positive:** the model rejects a borrower who would have repaid. This creates missed interest income and may harm customer acquisition.

Because these errors have different business costs, accuracy alone is not a sufficient evaluation metric. The project therefore combines standard classification metrics with an expected-loss framework.


## Data and Target Variable

The dataset contains loan amount, mortgage balance, property value, employment information, credit history, and debt-to-income variables. Missingness is meaningful in this dataset, so the final modelling scripts treat missing categorical values as an explicit `"Missing"` category and add missingness indicators for selected numerical fields.


In [ ]:
data_dictionary = pd.DataFrame([
    ("BAD", "Default indicator: 1 = default/severe delinquency, 0 = repaid"),
    ("LOAN", "Amount of loan approved"),
    ("MORTDUE", "Amount due on existing mortgage"),
    ("VALUE", "Current property value"),
    ("REASON", "Reason for loan request"),
    ("JOB", "Applicant job type"),
    ("YOJ", "Years at present job"),
    ("DEROG", "Number of major derogatory reports"),
    ("DELINQ", "Number of delinquent credit lines"),
    ("CLAGE", "Age of oldest credit line in months"),
    ("NINQ", "Number of recent credit inquiries"),
    ("CLNO", "Number of existing credit lines"),
    ("DEBTINC", "Debt-to-income ratio"),
], columns=["Variable", "Description"])

display(data_dictionary)


## Exploratory Analysis Snapshot

The EDA showed class imbalance, substantial missingness in selected variables, skewed financial variables, and visible relationships between default status and credit-history/debt-burden variables. The plots below are selected from the descriptive-analysis outputs.


In [ ]:
_table_7_1 = load_table("outputs/outputs_descr/target_balance.csv", "Target balance")
_table_7_2 = load_table("outputs/outputs_descr/missing_summary.csv", "Missing-value summary", sort_by="missing_pct", ascending=False)


In [ ]:
display_image("outputs/outputs_descr/target_distribution_bad.png", "Target distribution", width=650)
display_image("outputs/outputs_descr/numerical_distributions_by_bad.png", "Numerical distributions split by BAD", width=950)
display_image("outputs/outputs_descr/categorical_default_rates.png", "Categorical default-rate plots", width=850)


## Modelling Strategy

The modelling stage compares interpretable baselines and stronger non-linear models:

- Logistic regression
- Decision tree
- Random forest
- Gradient Boosting
- AdaBoost
- XGBoost

The revised scripts use a consistent, leakage-free approach:

- split data before fitting preprocessing steps;
- stratified train/test split with `random_state=42`;
- median imputation for numerical predictors;
- explicit `"Missing"` category for categorical predictors;
- one-hot encoding for `REASON` and `JOB`;
- model-specific tuning using cross-validation.


## Main Predictive Performance

The main predictive comparison uses the saved test-set metrics from the model scripts. The project has generally prioritised ROC-AUC because it measures ranking quality across thresholds, while F1, precision, and recall help assess the positive class `BAD = 1`.


In [ ]:
metric_columns = [
    "model", "accuracy", "precision_BAD_1", "recall_BAD_1", "f1_BAD_1", "roc_auc", "average_precision", "false_negatives", "false_positives"
]
comparison = load_table(
    "outputs/outputs_xg/xgb_comparison_with_previous_models.csv",
    "Main model comparison from the XGBoost script",
    columns=metric_columns,
    sort_by="roc_auc",
    ascending=False,
)


**Interpretation.** XGBoost is the strongest model on the main predictive metrics, with the highest ROC-AUC and F1 among the saved model comparisons. Gradient Boosting is very close and has slightly higher recall in the selected-threshold comparison, which means the ranking is not completely one-dimensional. This is why the later decision analysis is useful: it connects model errors to business costs.


## Confusion Matrices and Classification Trade-Offs

Confusion matrices show the practical consequences of a threshold. For default prediction, false negatives are especially important because they represent likely defaulters who would be approved by the model.


In [ ]:
display_image("outputs/outputs_logit/improved_confusion_matrix.png", "Logistic regression confusion matrix", width=520)
display_image("outputs/outputs_tree/tuned_tree_confusion_matrix.png", "Decision tree confusion matrix", width=520)
display_image("outputs/outputs_rf/tuned_rf_confusion_matrix.png", "Random forest confusion matrix", width=520)
display_image("outputs/outputs_gb/tuned_gb_confusion_matrix.png", "Gradient Boosting confusion matrix", width=520)
display_image("outputs/outputs_ada/tuned_ada_confusion_matrix.png", "AdaBoost confusion matrix", width=520)
display_image("outputs/outputs_xg/tuned_xgb_confusion_matrix.png", "XGBoost confusion matrix", width=520)


**Interpretation.** The stronger ensemble models reduce missed defaulters relative to simpler baselines, but threshold choice changes the balance between recall and precision. A lower threshold catches more defaulters, but also rejects more borrowers who would have repaid.


## ROC-AUC and Threshold Performance

ROC and precision-recall curves show how model performance changes across thresholds. The ensemble models have stronger separation between defaulters and non-defaulters than the simpler baselines.


In [ ]:
display_image("outputs/outputs_xg/xgb_roc_curve.png", "XGBoost ROC curve", width=700)
display_image("outputs/outputs_xg/xgb_precision_recall_curve.png", "XGBoost precision-recall curve", width=700)
display_image("outputs/outputs_rf/rf_roc_curve.png", "Random forest ROC curve", width=700)
display_image("outputs/outputs_gb/gb_roc_curve.png", "Gradient Boosting ROC curve", width=700)


In [ ]:
_table_18_1 = load_table("outputs/outputs_xg/xgb_threshold_analysis.csv", "XGBoost threshold analysis", sort_by="f1_BAD_1", ascending=False, n=10)


**Interpretation.** The model probability threshold should be treated as a business decision parameter, not a fixed statistical default. The best F1 threshold is not necessarily the best expected-loss threshold, because the economic cost of a false negative is much larger than the cost of a false positive in the baseline lending scenario.


## Expected-Loss Analysis

The expected-loss analysis translates model errors into economic costs:

- False negative cost: borrower defaults after being approved. Cost equals unrecovered loan value.
- False positive cost: borrower would have repaid but is rejected. Cost equals missed interest income.

The baseline assumptions are:

- 40% recovery rate
- 60% loss-given-default
- 4% opportunity cost on rejected good loans


In [ ]:
_table_21_1 = load_table("outputs/outputs_loss/loss_baseline_expected_loss.csv", "Expected loss at default 0.50 threshold", sort_by="total_expected_loss", ascending=True)
_table_21_2 = load_table("outputs/outputs_loss/loss_optimised_expected_loss.csv", "Expected loss after threshold optimisation", sort_by="total_expected_loss", ascending=True)
_table_21_3 = load_table("outputs/outputs_loss/loss_threshold_optimisation_summary.csv", "Threshold optimisation summary", sort_by="optimised_expected_loss", ascending=True)


In [ ]:
display_image("outputs/outputs_loss/loss_baseline_expected_loss.png", "Expected loss at default 0.50 threshold", width=750)
display_image("outputs/outputs_loss/loss_optimised_expected_loss.png", "Expected loss after threshold optimisation", width=750)
display_image("outputs/outputs_loss/loss_by_threshold.png", "Expected loss across thresholds", width=900)


**Interpretation.** XGBoost has the lowest expected loss in the baseline comparison and remains the lowest after threshold optimisation. Threshold optimisation materially lowers expected loss because the default threshold of `0.50` is not aligned with asymmetric lending costs. The economic analysis therefore adds a practical decision layer beyond standard predictive metrics.


## Robustness Checks

The robustness analysis varies recovery-rate and interest-rate assumptions. This tests whether the preferred model depends heavily on one narrow economic scenario.


In [ ]:
_table_25_1 = load_table("outputs/outputs_loss/loss_robustness_winner_counts.csv", "Best-model frequency across robustness grid")
_table_25_2 = load_table("outputs/outputs_loss/loss_robustness_best_model_by_assumption.csv", "Best model by recovery-rate / interest-rate assumption", n=15)


In [ ]:
display_image("outputs/outputs_loss/robustness_best_model_heatmap.png", "Best model under alternative economic assumptions", width=850)


**Interpretation.** The preferred model is fairly robust, but not universal. XGBoost wins most assumption combinations, while Random Forest and Gradient Boosting become preferred under some recovery-rate and opportunity-cost settings. This is a useful result: model selection should be reviewed under business assumptions rather than treated as purely technical.


## SHAP Model Interpretation

The SHAP analysis explains the best-performing main predictive model, which is XGBoost by ROC-AUC. This analysis is separate from the expected-loss threshold optimisation: it explains the predictive model itself.

For this XGBoost setup, SHAP values are reported in raw-margin / log-odds space. Positive SHAP values push predictions toward higher default risk; negative values push predictions toward lower default risk.


In [ ]:
_table_29_1 = load_table("outputs/outputs_shap/shap_model_selection_metrics.csv", "Model selection metrics for SHAP", columns=["model_label", "accuracy", "precision_BAD_1", "recall_BAD_1", "f1_BAD_1", "roc_auc", "average_precision"])
_table_29_2 = load_table("outputs/outputs_shap/shap_feature_importance.csv", "Top SHAP features", sort_by="mean_abs_shap", ascending=False, n=15, columns=["rank", "feature", "clean_feature_label", "mean_abs_shap", "mean_shap"])


In [ ]:
display_image("outputs/outputs_shap/shap_bar_mean_absolute.png", "SHAP mean absolute importance", width=800)
display_image("outputs/outputs_shap/shap_summary_beeswarm.png", "SHAP beeswarm summary", width=900)


In [ ]:
display_image("outputs/outputs_shap/shap_dependence_DEBTINC.png", "SHAP dependence: DEBTINC", width=700)
display_image("outputs/outputs_shap/shap_dependence_DEBTINC_missing.png", "SHAP dependence: DEBTINC_missing", width=700)
display_image("outputs/outputs_shap/shap_dependence_CLAGE.png", "SHAP dependence: CLAGE", width=700)
display_image("outputs/outputs_shap/shap_waterfall_high_risk_observation.png", "SHAP waterfall: high-risk observation", width=750)


**Interpretation.** The strongest drivers are debt-to-income burden, missing DTI information, credit age, delinquencies, loan amount, mortgage due, number of credit lines, derogatory reports, property value, and years on job. These are predictive associations rather than causal claims.


## Main Conclusions

1. **Best main predictive model:** XGBoost has the highest ROC-AUC and strongest overall balance of predictive metrics among the saved model comparisons.
2. **Best expected-loss model:** XGBoost also has the lowest expected loss under the baseline economic assumptions after threshold optimisation.
3. **Key trade-off:** lowering the classification threshold catches more likely defaulters but increases false positives. This can still reduce total expected loss when false negatives are much more expensive.
4. **Robustness:** XGBoost remains preferred across most recovery-rate and interest-rate scenarios, but Random Forest and Gradient Boosting win under some assumptions.
5. **Interpretability:** SHAP shows that the model relies mainly on debt burden, missing debt information, credit history, delinquencies, collateral-related variables, and loan exposure.


## Recommendations

A lender should use the XGBoost model as a decision-support benchmark, not as an automatic approval system. The operating threshold should be selected using explicit business assumptions about default losses, recovery rates, and opportunity costs.

Recommended practical approach:

- Use XGBoost as the primary predictive model.
- Set the decision threshold based on expected-loss analysis rather than defaulting to `0.50`.
- Monitor false negatives closely because missed defaulters are costly.
- Use SHAP explanations for model governance and human review.
- Reassess the threshold when recovery rates, loan pricing, or portfolio risk appetite change.


## Limitations and Possible Extensions

Important limitations:

- The dataset is historical and may not represent current borrower behaviour.
- Model outputs are predictive associations, not causal explanations.
- Fairness and compliance analysis has not yet been fully developed.
- Probability calibration should be assessed before production use.
- Threshold optimisation here uses the held-out test set for business analysis; production deployment should use validation or cross-validation policy tuning.

Possible extensions:

- Probability calibration using Platt scaling or isotonic regression.
- Fairness analysis across protected or proxy-sensitive variables.
- Cost-sensitive training rather than post-training threshold optimisation.
- Stability testing over time.
- A lightweight dashboard for comparing model decisions and economic assumptions.


## Appendix: Script Outputs Used

This notebook uses outputs generated by the following scripts:

- `project_descr.py`: descriptive analysis and EDA outputs
- `project_logit.py`: logistic regression outputs
- `project_tree.py`: decision tree outputs
- `project_rf.py`: random forest outputs
- `project_gb.py`: Gradient Boosting outputs
- `project_ada.py`: AdaBoost outputs
- `project_xg.py`: XGBoost outputs
- `project_loss.py`: expected-loss and robustness outputs
- `project_shap.py`: SHAP interpretation outputs

All paths in this notebook are relative to the project root.
